# Backbone Experiment: Audio Spectrogram Transformer (AST)

> **⚠️ Status: abandoned — training never completed on the available hardware.**\
> The setup, data prep, and model construction (Sections 1–6.4) all execute successfully, but the training loop in Section 6.4 could not finish even a single epoch under any configuration we tried (OOM at default batch size, then 60+ minutes per epoch with gradient checkpointing + bf16 mixed precision, then a VSCode kernel crash at hour-2 of one attempt). See **Section 6.5 — Outcome and limitations** for the full diagnosis. The companion notebook [`mel_pann.ipynb`](../Mel/mel_pann.ipynb) was created as the audio-pretrained alternative that *did* fit on this hardware.

Fine-tunes [`MIT/ast-finetuned-audioset-10-10-0.4593`](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593) — an Audio Spectrogram Transformer pretrained on AudioSet (2M+ audio clips) — on PSG-Audio for binary apnea detection.

Companion experiment to [`mel_cnn.ipynb`](../Mel/mel_cnn.ipynb) (ResNet18 baseline). Same data prep, splits, focal loss, and training loop; the only thing that changes is the backbone. The intent was to isolate the impact of swapping an ImageNet-pretrained CNN for an AudioSet-pretrained transformer.

## Why AST (the original motivation)
- **Pretrained on audio**, not natural images — early layers already understand spectrogram structure
- **16 kHz native** — matches our sample rate exactly
- **128 mel bins / hop=10ms** — standard AST config; we replicate it on GPU inside the model
- **Variable-length input** handled by pad/truncate to 1024 frames (~10.24 s)

In practice, AST's 1024-frame self-attention is quadratic in memory and does not fit on an 8 GB laptop GPU at any reasonable training speed. The full diagnosis of why is in Section 6.5.

## 1. Setup

Imports, device, and reproducibility seeds. Requires `transformers` — install with `pip install transformers` if missing.

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchaudio

from transformers import ASTForAudioClassification

# Shared dataset class (importable so DataLoader workers can pickle it on Windows)
from audio_datasets import RawWaveformDataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## 2. Configuration

Paths and hyperparameters. Notable differences vs `mel_cnn.ipynb`:
- Smaller `BATCH_SIZE` (AST ~85M params; doesn't fit at 128)
- Lower `LEARNING_RATE` (fine-tuning a pretrained transformer)
- Fewer `NUM_EPOCHS` (AST converges quickly from a strong pretrained start)
- No precomputed feature directory — AST computes its own features in `forward()`

In [3]:
# === Dataset ===
DATASET_ROOT = Path(r"C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\Dataset\Audio Dataset")
PSG_DIR      = DATASET_ROOT / "PSG-AUDIO" / "APNEA_EDF"
AP_TYPES_DIR = DATASET_ROOT / "APNEA_types"

# === Artifact output ===
PROJECT_ROOT = Path.cwd().resolve().parents[4]
ARTIFACT_DIR   = PROJECT_ROOT / "experiments" / "artifacts" / "audio" / "feature_representation" / "mel_ast"
CHECKPOINT_DIR = ARTIFACT_DIR / "checkpoints"
RESULTS_DIR    = ARTIFACT_DIR / "results"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# === Audio ===
SAMPLE_RATE = 16000

# === Apnea type mapping ===
APNEA_TYPE_NAMES = {0: "non-apnea", 1: "OSA", 2: "CSA", 3: "Mixed/Hypopnea", 4: "Other"}

# === AST params (must match the pretrained checkpoint's expectations) ===
AST_MODEL_NAME = "MIT/ast-finetuned-audioset-10-10-0.4593"
AST_N_MELS     = 128
AST_N_FFT      = 400      # 25 ms window @ 16 kHz
AST_HOP_LENGTH = 160      # 10 ms hop @ 16 kHz
AST_TARGET_FRAMES = 1024  # AST's max_length; pad/truncate to this
# Normalization stats from AST's feature extractor
AST_MEAN = -4.2677393
AST_STD  = 4.5689974

# === DataLoader ===
# bs=16 + bf16 autocast (in train/validate utilities) + gradient checkpointing
# in the model. bf16 halves activation memory, which lets us double the batch
# without OOM. Kept checkpointing on as a safety net — disable later if memory
# permits and you want more speed.
BATCH_SIZE   = 16
NUM_WORKERS  = 4
WAVE_CACHE_SIZE = 16

# === Loss ===
FOCAL_GAMMA = 2.0

# === Training ===
SEED                = 42
NUM_EPOCHS          = 8
LEARNING_RATE       = 1e-5      # 10x smaller than mel_cnn — fine-tuning a pretrained transformer
WEIGHT_DECAY        = 1e-3
DROPOUT             = 0.3       # AST already has internal dropout; modest head dropout
GRAD_CLIP           = 1.0
EARLY_STOP_PATIENCE = 3
LR_SCHED_PATIENCE   = 1
NUM_CLASSES         = 2
CLASS_NAMES         = ["non-apnea", "apnea"]

# === Reproducibility ===
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

print("Artifact dir:", ARTIFACT_DIR)
print("PSG dir exists:", PSG_DIR.exists())

Artifact dir: C:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\experiments\artifacts\audio\feature_representation\mel_ast
PSG dir exists: True


## 3. Data preparation

### 3.1 Build records DataFrame

Same flat-record layout as `mel_cnn.ipynb` — one row per segment, columns: `subject_id`, `segment_idx`, `label`, `apnea_type`, `file_type`.

In [4]:
def build_records(psg_dir, ap_types_dir):
    """Build (subject_id, segment_idx, label, apnea_type, file_type) records
    from npy headers without loading audio payloads."""
    records = []
    subjects = sorted(os.listdir(psg_dir))

    for subj in subjects:
        subj_path  = psg_dir / subj
        ap_file    = subj_path / f"{subj}_ap.npy"
        nap_file   = subj_path / f"{subj}_nap.npy"
        types_file = ap_types_dir / f"{subj}_ap_types.npy"

        if ap_file.exists():
            ap_types = np.load(types_file, allow_pickle=False) if types_file.exists() else None
            ap_mm = np.load(ap_file, mmap_mode="r")
            n_ap = ap_mm.shape[0]
            del ap_mm
            for i in range(n_ap):
                records.append({
                    "subject_id": subj, "segment_idx": i, "label": 1,
                    "apnea_type": int(ap_types[i]) if ap_types is not None else -1,
                    "file_type": "ap",
                })

        if not nap_file.exists():
            continue
        nap_mm = np.load(nap_file, mmap_mode="r")
        n_nap = nap_mm.shape[0]
        del nap_mm
        for i in range(n_nap):
            records.append({
                "subject_id": subj, "segment_idx": i, "label": 0,
                "apnea_type": 0, "file_type": "nap",
            })

    return pd.DataFrame(records)


df = build_records(PSG_DIR, AP_TYPES_DIR)
print("Total records:", len(df))
print(df["label"].value_counts())
df.head()

Total records: 103210
label
1    64585
0    38625
Name: count, dtype: int64


,subject_id,segment_idx,label,apnea_type,file_type
0,00000995-100507,0,1,1,ap
1,00000995-100507,1,1,1,ap
2,00000995-100507,2,1,1,ap
3,00000995-100507,3,1,1,ap
4,00000995-100507,4,1,1,ap


### 3.2 Subject-wise stratified split

Bucket subjects into quartiles by per-subject apnea fraction, then split within each bucket. Same recipe as `mel_cnn.ipynb` so the test set is identical (random_state=42), enabling apples-to-apples comparison.

In [5]:
def create_subjectwise_splits(df, subject_col="subject_id", train_size=0.70, val_size=0.15, test_size=0.15, random_state=42, n_strata=4):
    assert abs(train_size + val_size + test_size - 1.0) < 1e-8

    per_subj = (
        df.groupby(subject_col)["label"]
          .agg(["sum", "count"])
          .rename(columns={"sum": "n_apnea", "count": "n_total"})
    )
    per_subj["ap_frac"] = per_subj["n_apnea"] / per_subj["n_total"]
    per_subj["stratum"] = pd.qcut(per_subj["ap_frac"], q=n_strata, labels=False, duplicates="drop")

    train_subj, val_subj, test_subj = [], [], []
    val_ratio_of_temp = val_size / (val_size + test_size)

    for stratum, group in per_subj.groupby("stratum"):
        subjects = sorted(group.index.tolist())
        if len(subjects) < 3:
            train_subj.extend(subjects)
            continue
        tr, temp = train_test_split(subjects, test_size=(1 - train_size), random_state=random_state, shuffle=True)
        if len(temp) < 2:
            train_subj.extend(tr); val_subj.extend(temp); continue
        va, te = train_test_split(temp, test_size=(1 - val_ratio_of_temp), random_state=random_state, shuffle=True)
        train_subj.extend(tr); val_subj.extend(va); test_subj.extend(te)

    train_df = df[df[subject_col].isin(train_subj)].reset_index(drop=True)
    val_df   = df[df[subject_col].isin(val_subj)].reset_index(drop=True)
    test_df  = df[df[subject_col].isin(test_subj)].reset_index(drop=True)

    return train_df, val_df, test_df, train_subj, val_subj, test_subj


train_df, val_df, test_df, train_subjects, val_subjects, test_subjects = create_subjectwise_splits(df)

print(f"Train: {len(set(train_df['subject_id']))} subj, {len(train_df)} samples ({(train_df['label']==1).mean():.1%} apnea)")
print(f"Val  : {len(set(val_df['subject_id']))} subj, {len(val_df)} samples ({(val_df['label']==1).mean():.1%} apnea)")
print(f"Test : {len(set(test_df['subject_id']))} subj, {len(test_df)} samples ({(test_df['label']==1).mean():.1%} apnea)")

# Save for reproducibility
with open(RESULTS_DIR / "split_subjects.json", "w") as f:
    json.dump({
        "train_subjects": sorted(train_subjects),
        "val_subjects":   sorted(val_subjects),
        "test_subjects":  sorted(test_subjects),
    }, f, indent=2)

Train: 133 subj, 70430 samples (61.9% apnea)
Val  : 29 subj, 14822 samples (62.1% apnea)
Test : 30 subj, 17958 samples (65.8% apnea)


### 3.3 Class weights (focal-loss alpha)

Used as the `alpha` term in focal loss. Computed from train split only.

In [6]:
def compute_class_weights(train_df, label_col="label", num_classes=2):
    counts = train_df[label_col].value_counts().sort_index()
    total  = len(train_df)
    weights = torch.tensor([total / (num_classes * counts[c]) for c in range(num_classes)], dtype=torch.float32)
    print("Class counts (train):", counts.to_dict())
    print("Class weights:", weights.tolist())
    return weights


class_weights = compute_class_weights(train_df)

Class counts (train): {0: 26858, 1: 43572}
Class weights: [1.3111549615859985, 0.8082025051116943]


## 4. Datasets and DataLoaders

`RawWaveformDataset` (in `audio_datasets.py`) loads raw audio directly from the `*_ap.npy` / `*_nap.npy` files. AST does its own feature extraction inside `forward()`, so there's no precompute step here.

In [7]:
train_dataset = RawWaveformDataset(train_df, PSG_DIR, max_cache=WAVE_CACHE_SIZE)
val_dataset   = RawWaveformDataset(val_df,   PSG_DIR, max_cache=WAVE_CACHE_SIZE)
test_dataset  = RawWaveformDataset(test_df,  PSG_DIR, max_cache=WAVE_CACHE_SIZE)

pin = torch.cuda.is_available()
persist = NUM_WORKERS > 0
g = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=pin,
    persistent_workers=persist, generator=g,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=persist,
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=pin, persistent_workers=persist,
)

# Sanity check
waves, labels = next(iter(train_loader))
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Batch waveform shape: {waves.shape} | dtype: {waves.dtype}")
print(f"Batch labels: {labels[:10].tolist()}")

Train: 70430 | Val: 14822 | Test: 17958
Batch waveform shape: torch.Size([16, 160000]) | dtype: torch.float32
Batch labels: [1, 1, 1, 0, 1, 1, 1, 0, 1, 1]


## 5. Model

AST backbone with replaced 2-class head. Mel extraction (matched to AST's training-time config) lives inside `forward()` so it runs on GPU without a precomputed cache.

The pretrained weights expect log-mel power spectrograms normalized with the AudioSet stats (`AST_MEAN`, `AST_STD` × 2). Replicating that exactly is what makes the pretraining transfer.

In [8]:
class ASTClassifier(nn.Module):
    """AST backbone (AudioSet-pretrained) + 2-class head.

    Mel extraction runs on GPU inside forward() with parameters matched to
    AST's feature extractor: 128 mels, n_fft=400, hop=160, sr=16000. Output
    is padded/truncated to 1024 frames before the transformer.

    Gradient checkpointing is OFF: bf16 autocast + bs=16 fits comfortably,
    and checkpointing was causing the backward pass to recompute every
    transformer layer's activations — the main bottleneck behind 50+ min
    epochs. If memory OOMs, set grad_checkpointing=True and reduce BATCH_SIZE.
    """
    def __init__(self, num_classes=2, model_name=AST_MODEL_NAME, dropout=0.3,
                 grad_checkpointing=False):
        super().__init__()
        # Load AST with classifier replaced (ignore_mismatched_sizes drops the AudioSet head)
        self.ast = ASTForAudioClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True,
        )
        # Replace the AST classifier dense layer with our own (more dropout for safety)
        hidden_size = self.ast.config.hidden_size
        self.ast.classifier.dense = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, num_classes),
        )

        if grad_checkpointing:
            self.ast.config.use_cache = False
            self.ast.gradient_checkpointing_enable()

        # GPU-side mel feature extractor matching AST's expected input distribution
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE,
            n_fft=AST_N_FFT,
            win_length=AST_N_FFT,
            hop_length=AST_HOP_LENGTH,
            n_mels=AST_N_MELS,
            f_min=0.0,
            f_max=SAMPLE_RATE / 2,
            power=2.0,
        )
        self.register_buffer("_mean", torch.tensor(AST_MEAN))
        self.register_buffer("_std",  torch.tensor(AST_STD))
        self.target_frames = AST_TARGET_FRAMES

    def forward(self, x):
        # x: (B, T) raw waveform at SAMPLE_RATE
        mel = self.mel(x).clamp(min=1e-10).log()                # (B, n_mels, time)
        mel = (mel - self._mean) / (self._std * 2.0)            # AST normalization
        mel = mel.transpose(1, 2)                                # (B, time, n_mels)

        # Pad/truncate to target_frames (AST's positional embeddings are fixed-length)
        cur = mel.shape[1]
        if cur < self.target_frames:
            mel = nn.functional.pad(mel, (0, 0, 0, self.target_frames - cur))
        elif cur > self.target_frames:
            mel = mel[:, :self.target_frames, :]

        return self.ast(input_values=mel).logits


model = ASTClassifier(num_classes=NUM_CLASSES, dropout=DROPOUT, grad_checkpointing=False).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: ASTClassifier | trainable params: {n_params:,}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `527`.


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] ASTForAudioClassification LOAD REPORT from: MIT/ast-finetuned-audioset-10-10-0.4593
Key                     | Status   |                                                                                         
------------------------+----------+-----------------------------------------------------------------------------------------
classifier.dense.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527, 768]) vs model:torch.Size([2, 768])
classifier.dense.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([527]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
c:\Users\arshi\Desktop\AIS\Sem-2\Intelligent Sensing Systems\Practise Module\Project\ApneaSense\.venv\Lib\site-packages\torchaudio\functional\functional.py:581: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_

Model: ASTClassifier | trainable params: 86,190,338


## 6. Training

### 6.1 Loss, optimizer, scheduler

Same `FocalLoss` (γ=2.0, α=class weights) as `mel_cnn.ipynb` — keeps the loss isolated as a controlled variable. Lower learning rate (1e-5) than the from-scratch ResNet head, since we're fine-tuning a strong pretrained model.

In [9]:
class FocalLoss(nn.Module):
    """Multi-class focal loss with optional per-class alpha. See mel_cnn.ipynb for details."""
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        if alpha is not None:
            self.register_buffer("alpha", alpha)
        else:
            self.alpha = None

    def forward(self, logits, targets):
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = log_pt.exp()
        focal_weight = (1.0 - pt) ** self.gamma
        loss = -focal_weight * log_pt
        if self.alpha is not None:
            loss = self.alpha[targets] * loss
        return loss.mean()


criterion = FocalLoss(alpha=class_weights.to(device), gamma=FOCAL_GAMMA).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=LR_SCHED_PATIENCE)

print(f"Loss: FocalLoss(gamma={FOCAL_GAMMA}, alpha={class_weights.tolist()})")
print(f"Optimizer: AdamW | LR: {LEARNING_RATE} | WD: {WEIGHT_DECAY}")

Loss: FocalLoss(gamma=2.0, alpha=[1.3111549615859985, 0.8082025051116943])
Optimizer: AdamW | LR: 1e-05 | WD: 0.001


### 6.2 Train and validate utilities

In [10]:
def _epoch_metrics(loss_sum, n_samples, preds, labels):
    avg_loss = (loss_sum / n_samples).item()
    preds_np  = preds.cpu().numpy()
    labels_np = labels.cpu().numpy()
    acc      = (preds_np == labels_np).mean()
    macro_f1 = f1_score(labels_np, preds_np, average='macro')
    return avg_loss, acc, macro_f1


# bf16 autocast: forward + loss run in bfloat16 on Tensor Cores (~1.5-2x speed,
# ~50% less activation memory). bfloat16 has the same dynamic range as fp32
# so no GradScaler is needed (unlike fp16, which can underflow gradients).
# Backward, optimizer step, and reductions stay in fp32 automatically.
_AMP_DTYPE = torch.bfloat16


def train_one_epoch(model, loader, criterion, optimizer, device, grad_clip=None):
    model.train()
    loss_sum = torch.zeros(1, device=device)
    n_samples = 0
    all_preds, all_labels = [], []

    for inputs, labels in loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=_AMP_DTYPE):
            logits = model(inputs)
            loss   = criterion(logits, labels)
        loss.backward()
        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        bs = inputs.size(0)
        loss_sum += loss.detach().float() * bs
        n_samples += bs
        all_preds.append(torch.argmax(logits.detach(), dim=1))
        all_labels.append(labels)

    return _epoch_metrics(loss_sum, n_samples, torch.cat(all_preds), torch.cat(all_labels))


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    loss_sum = torch.zeros(1, device=device)
    n_samples = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=_AMP_DTYPE):
                logits = model(inputs)
                loss   = criterion(logits, labels)
            bs = inputs.size(0)
            loss_sum += loss.float() * bs
            n_samples += bs
            all_preds.append(torch.argmax(logits, dim=1))
            all_labels.append(labels)

    return _epoch_metrics(loss_sum, n_samples, torch.cat(all_preds), torch.cat(all_labels))

### 6.3 Checkpoint helper

In [11]:
def save_checkpoint(path, model, optimizer, epoch, best_val_acc, best_val_f1):
    torch.save({
        "epoch"               : epoch,
        "model_state_dict"    : model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_acc"        : best_val_acc,
        "best_val_macro_f1"   : best_val_f1,
        "num_classes"         : NUM_CLASSES,
        "backbone"            : AST_MODEL_NAME,
    }, path)

BEST_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_mel_ast.pth"

### 6.4 Training loop

The intended recipe: train for up to `NUM_EPOCHS` with early stopping on val macro-F1, save the best-by-val-F1 checkpoint, and reload it for evaluation in the (originally planned) Section 7.

**This is where the experiment ended.** Across multiple attempts with different `BATCH_SIZE` / gradient-checkpointing / bf16 combinations, the training loop never produced a single complete epoch on an 8 GB laptop GPU. The cell below is left intact as documentation of what was attempted; it will OOM, hang, or run for hours-per-epoch depending on which configuration you try. See Section 6.5 below for the full failure narrative — including how each remediation we tried interacted with AST's quadratic attention.

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'train_macro_f1': [],
           'val_loss': [],   'val_acc': [],   'val_macro_f1': []}

best_val_f1 = -1.0
best_val_acc = -1.0
best_epoch = -1
epochs_no_improve = 0

for epoch in range(NUM_EPOCHS):
    print(f'\nEpoch [{epoch + 1}/{NUM_EPOCHS}]')
    print('-' * 50)

    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, criterion, optimizer, device, grad_clip=GRAD_CLIP,
    )
    val_loss, val_acc, val_f1 = validate_one_epoch(model, val_loader, criterion, device)
    scheduler.step(val_f1)

    history['train_loss'].append(train_loss); history['train_acc'].append(train_acc); history['train_macro_f1'].append(train_f1)
    history['val_loss'].append(val_loss);     history['val_acc'].append(val_acc);     history['val_macro_f1'].append(val_f1)

    print(f'Train \u2014 Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}')
    print(f'Val   \u2014 Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_epoch = epoch + 1
        save_checkpoint(BEST_CHECKPOINT_PATH, model, optimizer, epoch, best_val_acc, best_val_f1)
        print(f'  \u2713 New best saved (Val F1: {best_val_f1:.4f})')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f'  No improvement for {epochs_no_improve} epoch(s)')
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f'\nEarly stopping triggered after {epoch + 1} epochs.')
            break

actual_epochs = len(history['train_loss'])
print(f'\nBest epoch: {best_epoch} | Best Val F1: {best_val_f1:.4f} | Best Val Acc: {best_val_acc:.4f}')


Epoch [1/8]
--------------------------------------------------


## 6.5 Outcome and limitations

### What happened

Five attempts at training, each with a different mitigation, all failed:

| Attempt | Configuration | Result |
|---|---|---|
| 1 | `BATCH_SIZE=32`, fp32, no gradient checkpointing | **CUDA OOM** during the first backward pass |
| 2 | `BATCH_SIZE=8`, fp32, gradient checkpointing on | Ran but **>60 min/epoch**; interrupted manually |
| 3 | `BATCH_SIZE=16`, **bf16 autocast**, gradient checkpointing on | Still **~50 min/epoch**; interrupted manually |
| 4 | `BATCH_SIZE=16`, bf16, gradient checkpointing **off** | **134 min into epoch 1**, VSCode kernel crashed (system-level resource exhaustion) |
| 5 | After confirming Windows Search Indexer was hogging the disk: same as attempt 4 with disk freed | First epoch still running at the **2-hour mark**, then VSCode crashed again |

No checkpoint, no metrics, no curves were ever produced.

### Why AST didn't fit on this hardware

AST processes its 1024-frame mel spectrogram as a sequence of patches and feeds them through a 12-layer transformer. The dominant memory cost is **self-attention activations**, which scale **quadratically** in the input sequence length: `O(B · L²)` per layer, then accumulated for backward.

For our config (`L=1024`, 12 attention heads × 12 transformer layers, ~85M params, hidden size 768):

| Memory term | Approx. size at `BATCH_SIZE=32` |
|---|---|
| Model weights | ~340 MB (fp32) |
| Optimizer states (AdamW: 2× weights) | ~680 MB |
| Attention activation tensors (`Q·Kᵀ`) | **~30+ GB** |

The 8 GB VRAM on the laptop GPU has **no realistic path** to absorbing a 30+ GB activation budget. Reducing batch size, enabling bf16, and turning on gradient checkpointing each shave a fraction, but each comes with its own cost (slower training, recomputation overhead) — and combined they still left the run far slower than acceptable.

### Why mel_cnn and mel_pann didn't have this problem

- **`mel_cnn.ipynb` (ResNet18):** ~11M params, fully convolutional. Activations scale linearly in input dimensions, not quadratically. A batch of 128 fits in <2 GB.
- **`mel_pann.ipynb` (PANN Cnn14):** ~80M params (similar scale to AST) but **also fully convolutional** — no self-attention. Even with PANN's similar parameter count, activation memory at `BATCH_SIZE=32` is ~3-4 GB, well within 8 GB.

The lesson: **for training pretrained audio models on a laptop GPU, the architectural choice between CNN and transformer matters far more than the parameter count.** A transformer with the same parameter budget as a CNN can be 10× heavier on activation memory because of attention's quadratic scaling.

### Failure modes worth documenting

- **Gradient checkpointing has a real wall-clock cost.** Each backward pass recomputes every transformer layer's forward activations. For AST's 12 layers, that's ~2× total compute — turning a 30-min epoch into a 60+-min epoch.
- **bf16 autocast helps but isn't sufficient.** It halves activation memory and ~1.5-2x speeds up matmul on Tensor Cores, but doesn't change the *quadratic* scaling. Going from `O(L²)` to `O(L²)/2` is a constant-factor win, not a structural one.
- **System-level resource contention amplifies everything.** During one long-running attempt, Windows Search Indexer was reading 268 MB/s from disk (re-indexing recently-created mel cache files), starving the data loader and tripling effective epoch time. We disabled it via `Stop-Service WSearch` — necessary but not sufficient.
- **VSCode kernel crashes at hour-2.** The notebook environment isn't designed for sustained multi-hour training of large transformers. Pure-Python scripts running headless are more robust for that workload.

### What would make AST viable

- **Larger GPU.** Cloud T4/L4 (16 GB) or A100 (40 GB) would absorb the attention activations comfortably and run epochs in minutes.
- **A different transformer.** Audio MAE, BEATs, or longer-context-friendly variants with sparse/linear attention would scale better at `L=1024`.
- **Truncated sequences.** Cutting `AST_TARGET_FRAMES` from 1024 → 512 would cut attention memory 4×, but breaks the pretrained positional embeddings — likely a meaningful accuracy hit.
- **Layer freezing.** Freezing the lower 6-9 transformer blocks and fine-tuning only the head + last few blocks would reduce the activation graph that needs to be retained for backward. Not attempted here because we pivoted to PANN instead.

### Where to look for results

Since this experiment produced no metrics, the audio backbone comparison in the project lives in:

- **[`mel_cnn.ipynb`](../Mel/mel_cnn.ipynb)** — ResNet18, ImageNet-pretrained — the headline result (Test F1 ≈ 0.78, Test Acc ≈ 0.80).
- **[`mel_pann.ipynb`](../Mel/mel_pann.ipynb)** — PANN Cnn14, AudioSet-pretrained — the audio-pretrained alternative that fit on this hardware (Test F1 ≈ 0.75).
- **[`raw_waveform_1dcnn.ipynb`](../Raw_waveform_1dcnn/raw_waveform_1dcnn.ipynb)** — 1D CNN trained from scratch on raw audio — no-feature-engineering baseline (Test F1 ≈ 0.63).

This notebook stands as a documented attempt and a hardware-constraint case study, not as a comparable backbone result.